# Lilly — cut OCR crops (data prep, NOT a training pass)

This notebook runs the text detector over photographs and cuts the text regions
out as crops. It **does not train anything**, does not touch `lilly.pth`, and
does not go near the crop gate. Nothing here can replace the shipped reader.

It exists because cutting 419 photographs measured at ~76 s each on the owner's
8 GB Mac — about nine hours, while both Kaggle GPU slots sat empty. `docs/V2-BOUNDARIES.md`
already says where compute belongs: Kaggle computes, the Mac coordinates.

Output is `lilly-crops.zip`: the crops plus `labels.tsv`, which carries the
reader's own guess. That guess is **never shown to whoever labels the crops** —
`data/scripts/label_crops.py` builds blind sheets from the images alone, because
a proofreader agrees with a confident wrong answer far more often than a
transcriber independently produces it.

In [ ]:
# 1. Stop here unless the machine is actually set up
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)
print("network ok")

# Under /kaggle/working, so it is Output and outlives the log.
# agentic-kaggle-skill offload: the tee is stdout.txt, not a child fd Kaggle never sees.
TEE = Path("/kaggle/working/stdout.txt")

def run(*cmd, quiet=False):
    """Run a child process and put its output somewhere it can be found.

    Kaggle's log holds what this notebook process prints. A child process
    writing to its own stdout is not in it. Pass-7b and pass-7c both logged
    `$ python3 ... train_ocr.py` and then nothing whatsoever until the zip 21
    minutes later, and `python -u` on the child did not fix it, because the
    child's file descriptor never reaches the log to begin with. Both runs
    therefore ended with their before/after rates and the gate's own verdict
    unrecoverable, and pass-7c had to be re-measured on a laptop to discover it
    read photographs worse. Read the child's output here and reprint it, and
    tee it into Output so the numbers survive a dropped log too.
    """
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)

In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working
# Everything under /kaggle/working becomes Output. Training copies tens of
# thousands of PNGs into data/ocr/train and valid, and with the clone there too,
# `kaggle kernels output` spent 172 s on PNGs and git objects and never reached
# the weights zip at all. Only the zips below belong in Output.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "prepare_ocr_data.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zips")
from training.kaggle_offload import Offload
OFF = Offload("ocr-crops", os.environ.get("LILLY_RUN_ID", "ocr-crops"))
OFF.hardware(torch.cuda.get_device_name(0))

In [ ]:
# 3. Install what we need (~2 min)
# Not torch: Kaggle ships a GPU build and requirements.txt pins CPU wheels for
# the Mac. Replacing it wastes time and can break CUDA.
NEEDED = ["easyocr", "opencv-python-headless", "pillow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line and not line.startswith("--"):
        pins[line.split("==")[0].strip().lower()] = line
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# 4. Find the attached photographs, and refuse the ones that are the exam
import glob, json as _json

found = sorted({os.path.dirname(p) for p in
                glob.glob("/kaggle/input/**/*.jpg", recursive=True)})
assert found, ("Attach the Lilly OCR photo set as a Dataset input. "
               "scripts/kaggle_train.py uploads it as <user>/lilly-ocr-photos "
               "and attaches it automatically — launch from the laptop with "
               "`python3 scripts/kaggle_train.py ocr-crops`.")
PHOTOS = Path(found[0])
names = sorted(p.name for p in PHOTOS.glob("*.jpg"))
print(f"{len(names)} photographs in {PHOTOS}")

# The 40 scored photographs decide whether the reader improved. A crop cut from
# one of them turns the next score into recital: this project already lost every
# reader number it had published to exactly that, when 873 label texts sat on
# both sides at once. The laptop filters them before upload; this checks again,
# because a filter that is only applied in one place is a filter that will one
# day be skipped.
truth = _json.loads(Path("data/ocr/real-photos/truth.json").read_text(encoding="utf-8"))
scored = set(truth["photos"])
leak = sorted(set(names) & scored)
if leak:
    raise SystemExit(
        f"{len(leak)} of the attached photographs are in the 40 scored set "
        f"({', '.join(leak[:5])}...). Cutting training crops from them would make "
        f"the reader's own score recital. Re-upload without them.")
print(f"leak gate: 0 of {len(names)} are among the {len(scored)} scored photographs")

In [ ]:
# 5. Cut the crops (the whole point — detector inference, no training)
CROPS = SCRATCH / "crops2"          # scratch, not /kaggle/working: only the zip is Output
run(sys.executable, "training/prepare_ocr_data.py",
    "--photos", str(PHOTOS), "--crops", str(CROPS))

In [ ]:
# 6. Gate, then package
# A detector that loaded but found nothing still exits 0 and leaves an empty
# folder. Zipping that would send back a file that looks like a result and is
# not, which is the shape fail-stop rule 14 is about.
cut = sorted(CROPS.glob("*.png"))
labels = CROPS / "labels.tsv"
photos_seen = len({p.name.rsplit("_", 1)[0] for p in cut})
print(f"{len(cut):,} crops from {photos_seen} of {len(names)} photographs")

if len(cut) < 200:
    raise SystemExit(f"only {len(cut)} crops from {len(names)} photographs — the "
                     f"detector found almost nothing. Not packaging this.")
if photos_seen < len(names) // 2:
    raise SystemExit(f"crops came from only {photos_seen} of {len(names)} photographs — "
                     f"the run did not finish. Not packaging a partial cut.")
assert labels.is_file(), "labels.tsv was not written — nothing to label against"

run("zip", "-qrj", "/kaggle/working/lilly-crops.zip", str(CROPS))
out = Path("/kaggle/working/lilly-crops.zip")
size = out.stat().st_size
assert size > 1_000_000, f"lilly-crops.zip is only {size} bytes — the save did not happen"
print(f"lilly-crops.zip — {size / 1048576:.1f} MB, {len(cut):,} crops, in the Output tab")
OFF.metric("crops", len(cut)); OFF.metric("photographs", photos_seen)

**Done.** Fetch with `python3 scripts/kaggle_train.py ocr-crops --fetch`, unzip
into `data/ocr/crops2/`, then build the blind sheets:

```
python3 data/scripts/label_crops.py sheets --crops data/ocr/crops2
```

The second batch keeps its own sheets and answers directory. Sheet numbering is
only unique within one batch, so collecting batch 2 against batch 1's manifest
would silently attach the wrong text to the wrong picture.